# Sprint 4 - Experiment Tracking

Este notebook consolida los resultados del Sprint 4, actualiza el registro de experimentos, valida la carga del modelo final y deja trazabilidad desde los modelos baseline hasta el modelo final optimizado.

## 1. Importación de librerías y Carga del registro de experimentos

In [28]:
import os
import joblib
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

LOG_PATH = "../models/experiments_log.csv"
MODELS_DIR = "../models"
FINAL_MODEL_PATH = "../models/final_model.pkl"

## 2. Carga de insumos

Se cargan los datos de prueba, las etiquetas reales y el archivo `experiments_log.csv` generado al cierre del Sprint 3. Este archivo será actualizado con el modelo final del Sprint 4.

In [30]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

experiments_log = pd.read_csv(LOG_PATH)

print("✅ Insumos cargados correctamente")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Registros actuales en experiments_log:", len(experiments_log))

experiments_log

✅ Insumos cargados correctamente
X_test shape: (23841, 64)
y_test shape: (23841,)
Registros actuales en experiments_log: 16


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,...,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,sprint,tipo_modelo,dataset,path
0,dt,random_state=42,2026-05-05,0.816071,NaN,0.761476,0.754286,0.807445,0.821484,0.773278,...,0.820202,-0.011801,-0.014425,0.33,True,Rank 1 CV Recall. Sin overfitting. Seleccionad...,NaN,NaN,NaN,NaN
1,rf,random_state=42,2026-05-05,0.862808,NaN,0.752171,0.802576,0.926593,0.866994,0.769205,...,0.934606,-0.017034,-0.013239,4.33,True,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...,NaN,NaN,NaN,NaN
2,xgb,random_state=42,2026-05-05,0.846450,NaN,0.713676,0.775089,0.915453,0.845560,0.716031,...,0.916261,-0.002356,-0.001761,0.55,True,Rank 3 CV Recall. Gap mínimo de overfitting. E...,NaN,NaN,NaN,NaN
3,lr,"random_state=42, max_iter=1000",2026-05-05,0.812653,NaN,0.618413,0.709945,0.860963,0.784866,0.615115,...,0.863555,0.003298,0.003566,4.30,False,CV Recall bajo (0.618). Modelo lineal insufici...,NaN,NaN,NaN,NaN
4,gb,random_state=42,2026-05-05,0.818368,NaN,0.616122,0.715516,0.885453,0.814479,0.611495,...,0.883982,0.004628,0.004285,5.89,False,CV Recall bajo (0.616). Más lento que RF/XGB s...,NaN,NaN,NaN,NaN
5,NN,default,2026-05-05,0.808993,NaN,0.614821,0.703766,0.873830,NaN,0.647585,...,0.885184,-0.032763,-0.022261,6.45,False,CV Recall más bajo (0.615). Mayor tiempo de en...,NaN,NaN,NaN,NaN
6,tuned_rf,"n_estimators=500, min_samples_split=10, min_sa...",2026-05-08,NaN,NaN,0.713400,0.785000,0.924300,NaN,0.724100,...,0.930700,-0.010700,-0.008300,NaN,False,RF tuneado con RandomizedSearchCV. Mejora reca...,4.0,NaN,NaN,NaN
7,tuned_xgb,"n_estimators=500, max_depth=5, learning_rate=0...",2026-05-08,NaN,NaN,0.766300,0.804100,0.927300,NaN,0.775300,...,0.935700,-0.009100,-0.008400,NaN,True,XGB tuneado con RandomizedSearchCV. MODELO FIN...,4.0,NaN,NaN,NaN
8,ensemble_hv,"voting=hard, estimators=[tuned_rf, tuned_xgb]",2026-05-08,NaN,NaN,0.692300,0.780600,NaN,NaN,0.708300,...,NaN,-0.016000,-0.012300,NaN,False,"Hard Voting. No soporta predict_proba, AUC no ...",4.0,NaN,NaN,NaN
9,ensemble_sv,"voting=soft, weights=[3,3], estimators=[tuned_...",2026-05-08,NaN,NaN,0.749500,0.802600,0.929900,NaN,0.760300,...,0.937000,-0.010800,-0.010200,NaN,False,Soft Voting con pesos iguales. AUC más alto de...,4.0,NaN,NaN,NaN


In [ ]:
## 2. Normalización de columnas del registro

Se verifica que el archivo `experiments_log.csv` contenga las columnas necesarias para registrar resultados del Sprint 4, incluyendo tipo de modelo, dataset utilizado y ruta del artefacto.

In [31]:
required_columns = [
    "model", "params", "date",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc",
    "test_accuracy", "test_recall", "test_f1", "test_roc_auc",
    "recall_gap", "f1_gap", "train_time_s",
    "selected", "notes",
    "tipo_modelo", "dataset", "path"
]

for col in required_columns:
    if col not in experiments_log.columns:
        experiments_log[col] = None

experiments_log = experiments_log[required_columns]
experiments_log.to_csv(LOG_PATH, index=False)

print("✅ experiments_log.csv normalizado correctamente")
experiments_log.head()

✅ experiments_log.csv normalizado correctamente


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,dt,random_state=42,2026-05-05,0.816071,NaN,0.761476,0.754286,0.807445,0.821484,0.773278,0.768712,0.820202,-0.011801,-0.014425,0.33,True,Rank 1 CV Recall. Sin overfitting. Seleccionad...,NaN,NaN,NaN
1,rf,random_state=42,2026-05-05,0.862808,NaN,0.752171,0.802576,0.926593,0.866994,0.769205,0.815815,0.934606,-0.017034,-0.013239,4.33,True,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...,NaN,NaN,NaN
2,xgb,random_state=42,2026-05-05,0.846450,NaN,0.713676,0.775089,0.915453,0.845560,0.716031,0.776850,0.916261,-0.002356,-0.001761,0.55,True,Rank 3 CV Recall. Gap mínimo de overfitting. E...,NaN,NaN,NaN
3,lr,"random_state=42, max_iter=1000",2026-05-05,0.812653,NaN,0.618413,0.709945,0.860963,0.784866,0.615115,0.706379,0.863555,0.003298,0.003566,4.30,False,CV Recall bajo (0.618). Modelo lineal insufici...,NaN,NaN,NaN
4,gb,random_state=42,2026-05-05,0.818368,NaN,0.616122,0.715516,0.885453,0.814479,0.611495,0.711231,0.883982,0.004628,0.004285,5.89,False,CV Recall bajo (0.616). Más lento que RF/XGB s...,NaN,NaN,NaN


In [32]:
required_columns = [
    "model", "params", "date",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc",
    "test_accuracy", "test_recall", "test_f1", "test_roc_auc",
    "recall_gap", "f1_gap", "train_time_s",
    "selected", "notes",
    "tipo_modelo", "dataset", "path"
]

for col in required_columns:
    if col not in experiments_log.columns:
        experiments_log[col] = None

experiments_log = experiments_log[required_columns]
experiments_log.to_csv(LOG_PATH, index=False)

print("✅ experiments_log.csv normalizado correctamente")
experiments_log.head()

✅ experiments_log.csv normalizado correctamente


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,dt,random_state=42,2026-05-05,0.816071,NaN,0.761476,0.754286,0.807445,0.821484,0.773278,0.768712,0.820202,-0.011801,-0.014425,0.33,True,Rank 1 CV Recall. Sin overfitting. Seleccionad...,NaN,NaN,NaN
1,rf,random_state=42,2026-05-05,0.862808,NaN,0.752171,0.802576,0.926593,0.866994,0.769205,0.815815,0.934606,-0.017034,-0.013239,4.33,True,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...,NaN,NaN,NaN
2,xgb,random_state=42,2026-05-05,0.846450,NaN,0.713676,0.775089,0.915453,0.845560,0.716031,0.776850,0.916261,-0.002356,-0.001761,0.55,True,Rank 3 CV Recall. Gap mínimo de overfitting. E...,NaN,NaN,NaN
3,lr,"random_state=42, max_iter=1000",2026-05-05,0.812653,NaN,0.618413,0.709945,0.860963,0.784866,0.615115,0.706379,0.863555,0.003298,0.003566,4.30,False,CV Recall bajo (0.618). Modelo lineal insufici...,NaN,NaN,NaN
4,gb,random_state=42,2026-05-05,0.818368,NaN,0.616122,0.715516,0.885453,0.814479,0.611495,0.711231,0.883982,0.004628,0.004285,5.89,False,CV Recall bajo (0.616). Más lento que RF/XGB s...,NaN,NaN,NaN


## 3. Verificación de modelos y artefactos disponibles

Se identifican los modelos y artefactos almacenados en la carpeta `models/`, incluyendo archivos `.pkl` y archivos versionados con DVC (`.dvc`).

In [33]:
artifacts = sorted([
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pkl") or f.endswith(".pkl.dvc") or f.endswith(".dvc")
])

print("Modelos y artefactos disponibles en /models:\n")

for f in artifacts:
    print("-", f)

Modelos y artefactos disponibles en /models:

- baseline_NN.pkl
- baseline_NN.pkl.dvc
- baseline_dt.pkl
- baseline_dt.pkl.dvc
- baseline_gb.pkl
- baseline_gb.pkl.dvc
- baseline_lr.pkl
- baseline_lr.pkl.dvc
- baseline_rf.pkl.dvc
- baseline_xgb.pkl
- baseline_xgb.pkl.dvc
- ensemble_hv.pkl.dvc
- ensemble_stack.pkl.dvc
- ensemble_sv.pkl.dvc
- final_model.pkl
- preprocessor.pkl
- preprocessor.pkl.dvc
- tuned_rf.pkl.dvc
- tuned_xgb.pkl.dvc


In [23]:
def infer_tipo_modelo(filename):
    if filename.startswith("baseline_"):
        return "baseline"
    elif filename.startswith("tuned_"):
        return "tuned"
    elif filename.startswith("ensemble_"):
        return "ensemble"
    elif filename == "final_model.pkl":
        return "final"
    else:
        return "unknown"


def infer_model_name(filename):
    name = filename.replace(".pkl", "")
    name = name.replace("baseline_", "")
    name = name.replace("tuned_", "")
    name = name.replace("ensemble_", "")
    return name


def get_model_params(model):
    try:
        return str(model.get_params())
    except Exception:
        return "No disponible"


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": None
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)[:, 1]
            metrics["test_roc_auc"] = roc_auc_score(y_test, y_proba)
        except Exception:
            metrics["test_roc_auc"] = None

    return metrics

## 4. Funciones auxiliares

Se definen funciones para evaluar el modelo final y registrar el resultado en `experiments_log.csv` sin sobrescribir los registros previos.

In [34]:
def evaluate_model(model, X_test, y_test):
    """
    Evalúa un modelo sobre el conjunto de prueba.
    """

    y_pred = model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": None
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)[:, 1]
            metrics["test_roc_auc"] = roc_auc_score(y_test, y_proba)
        except Exception as e:
            print("⚠️ No se pudo calcular ROC-AUC:", e)

    return metrics


def get_model_params(model):
    """
    Intenta recuperar los parámetros del modelo.
    """

    try:
        return str(model.get_params())
    except Exception:
        return "Parámetros no disponibles"


def save_experiment(result, log_path=LOG_PATH):
    """
    Agrega un experimento nuevo al experiments_log.csv sin borrar registros previos.
    """

    df_new = pd.DataFrame([result])

    if os.path.exists(log_path):
        df_old = pd.read_csv(log_path)

        for col in df_new.columns:
            if col not in df_old.columns:
                df_old[col] = None

        for col in df_old.columns:
            if col not in df_new.columns:
                df_new[col] = None

        df_new = df_new[df_old.columns]

        # Evitar duplicar el modelo final si ya fue registrado previamente
        if "path" in df_old.columns and result["path"] in df_old["path"].astype(str).values:
            print("⚠️ Este modelo ya estaba registrado en experiments_log.csv")
            return df_old

        df_final = pd.concat([df_old, df_new], ignore_index=True)

    else:
        df_final = df_new

    df_final.to_csv(log_path, index=False)

    print("✅ Experimento registrado correctamente")
    print("Total registros:", len(df_final))

    return df_final

In [25]:
def infer_tipo_modelo(filename):
    if filename.startswith("baseline_"):
        return "baseline"
    elif filename.startswith("tuned_"):
        return "tuned"
    elif filename.startswith("ensemble_"):
        return "ensemble"
    elif filename == "final_model.pkl":
        return "final"
    else:
        return "unknown"


def infer_model_name(filename):
    name = filename.replace(".pkl", "")
    name = name.replace("baseline_", "")
    name = name.replace("tuned_", "")
    name = name.replace("ensemble_", "")
    return name


def get_model_params(model):
    try:
        return str(model.get_params())
    except Exception:
        return "No disponible"


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": None
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)[:, 1]
            metrics["test_roc_auc"] = roc_auc_score(y_test, y_proba)
        except Exception:
            metrics["test_roc_auc"] = None

    return metrics

In [35]:
# Registrar modelos reales disponibles

model_files_to_register = [
    "tuned_rf.pkl",
    "tuned_xgb.pkl",
    "ensemble_hv.pkl",
    "ensemble_sv.pkl",
    "ensemble_stack.pkl",
    "final_model.pkl"
]

new_results = []

for filename in model_files_to_register:
    path = os.path.join(MODELS_DIR, filename)

    if not os.path.exists(path):
        print(f"⚠️ No se encontró {filename}. Si existe {filename}.dvc, ejecutar: dvc pull")
        continue

    model = joblib.load(path)
    metrics = evaluate_model(model, X_test, y_test)

    tipo_modelo = infer_tipo_modelo(filename)
    model_name = infer_model_name(filename)

    result = {
        "model": model_name,
        "params": get_model_params(model),
        "date": datetime.today().strftime("%Y-%m-%d"),

        "cv_accuracy": None,
        "cv_precision": None,
        "cv_recall": None,
        "cv_f1": None,
        "cv_roc_auc": None,

        "test_accuracy": round(metrics["test_accuracy"], 6),
        "test_recall": round(metrics["test_recall"], 6),
        "test_f1": round(metrics["test_f1"], 6),
        "test_roc_auc": round(metrics["test_roc_auc"], 6) if metrics["test_roc_auc"] is not None else None,

        "recall_gap": None,
        "f1_gap": None,
        "train_time_s": None,

        "selected": True if filename == "final_model.pkl" else False,
        "notes": f"Registro automático Sprint 4 desde {filename}",

        "tipo_modelo": tipo_modelo,
        "dataset": "global",
        "path": path
    }

    new_results.append(result)

print(f"✅ Modelos registrados desde archivos reales: {len(new_results)}")
pd.DataFrame(new_results)

⚠️ No se encontró tuned_rf.pkl. Si existe tuned_rf.pkl.dvc, ejecutar: dvc pull
⚠️ No se encontró tuned_xgb.pkl. Si existe tuned_xgb.pkl.dvc, ejecutar: dvc pull
⚠️ No se encontró ensemble_hv.pkl. Si existe ensemble_hv.pkl.dvc, ejecutar: dvc pull
⚠️ No se encontró ensemble_sv.pkl. Si existe ensemble_sv.pkl.dvc, ejecutar: dvc pull
⚠️ No se encontró ensemble_stack.pkl. Si existe ensemble_stack.pkl.dvc, ejecutar: dvc pull
✅ Modelos registrados desde archivos reales: 1


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,final_model,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,None,None,None,None,None,0.867371,0.775314,0.812544,0.935699,None,None,None,True,Registro automático Sprint 4 desde final_model...,final,global,../models\final_model.pkl


## 5. Validación del modelo final

Se carga el archivo `final_model.pkl` y se verifica que pueda generar predicciones sobre el conjunto de prueba. Esta validación confirma que el artefacto del modelo final es reutilizable.

In [9]:
result_stacking = {
    "model": "stacking_rf_xgb",
    "params": "base models: tuned_rf + tuned_xgb; meta learner: LogisticRegression",
    "date": datetime.today().strftime("%Y-%m-%d"),
    "cv_accuracy": None,
    "cv_precision": None,
    "cv_recall": 0.000,
    "cv_f1": 0.000,
    "cv_roc_auc": 0.000,
    "test_accuracy": None,
    "test_recall": None,
    "test_f1": None,
    "test_roc_auc": None,
    "recall_gap": None,
    "f1_gap": None,
    "train_time_s": None,
    "selected": False,
    "notes": "Stacking con RF tuneado y XGB tuneado; Logistic Regression como meta-learner",
    "tipo_modelo": "ensemble",
    "dataset": "global",
    "path": "../models/ensemble_stacking_rf_xgb.pkl"
}

experiments_log = save_experiment(result_stacking)
experiments_log.tail()

✅ Experimento registrado
Total registros: 15


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,...,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,sprint,tipo_modelo,dataset,path
10,ensemble_stack,"estimators=[tuned_xgb, tuned_rf], final_estima...",2026-05-08,NaN,NaN,0.7619,0.8054,0.9293,NaN,0.7745,...,0.9364,-0.0126,-0.0096,NaN,False,Stacking con meta-learner LR. F1 más alto pero...,4.0,NaN,NaN,NaN
11,rf,REEMPLAZAR_CON_BEST_PARAMS_RF,2026-05-08,NaN,NaN,0.0000,0.0000,0.0000,NaN,NaN,...,NaN,NaN,NaN,NaN,False,Random Forest tuneado en Sprint 4,NaN,tuned,global,../models/tuned_rf.pkl
12,xgb,REEMPLAZAR_CON_BEST_PARAMS_XGB,2026-05-08,NaN,NaN,0.0000,0.0000,0.0000,NaN,NaN,...,NaN,NaN,NaN,NaN,False,XGBoost tuneado en Sprint 4,NaN,tuned,global,../models/tuned_xgb.pkl
13,voting_rf_xgb,soft voting: tuned_rf + tuned_xgb,2026-05-08,NaN,NaN,0.0000,0.0000,0.0000,NaN,NaN,...,NaN,NaN,NaN,NaN,False,Ensamble Soft Voting con RF tuneado y XGB tuneado,NaN,ensemble,global,../models/ensemble_voting_rf_xgb.pkl
14,stacking_rf_xgb,base models: tuned_rf + tuned_xgb; meta learne...,2026-05-08,None,None,0.0000,0.0000,0.0000,None,None,...,None,None,None,None,False,Stacking con RF tuneado y XGB tuneado; Logisti...,None,ensemble,global,../models/ensemble_stacking_rf_xgb.pkl


In [36]:
if os.path.exists(FINAL_MODEL_PATH):
    final_model = joblib.load(FINAL_MODEL_PATH)
    sample_predictions = final_model.predict(X_test.head())

    print("✅ final_model.pkl carga correctamente")
    print("Predicciones de prueba:", sample_predictions)
else:
    raise FileNotFoundError("No se encontró ../models/final_model.pkl")

✅ final_model.pkl carga correctamente
Predicciones de prueba: [1 0 0 1 1]


## 6. Evaluación del modelo final

Se calculan las métricas principales del modelo final sobre el conjunto de prueba: accuracy, recall, F1-score y ROC-AUC.

In [37]:
final_metrics = evaluate_model(final_model, X_test, y_test)

print("Métricas del modelo final:")
for metric, value in final_metrics.items():
    print(metric, ":", round(value, 6) if value is not None else None)

Métricas del modelo final:
test_accuracy : 0.867371
test_recall : 0.775314
test_f1 : 0.812544
test_roc_auc : 0.935699


In [12]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   --- ------------------------------------ 8.1/101.7 MB 42.1 MB/s eta 0:00:03
   -------- ------------------------------- 21.8/101.7 MB 55.1 MB/s eta 0:00:02
   --------------- ------------------------ 40.1/101.7 MB 65.4 MB/s eta 0:00:01
   --------------------- ------------------ 55.6/101.7 MB 69.4 MB/s eta 0:00:01
   --------------------------- ------------ 70.3/101.7 MB 66.9 MB/s eta 0:00:01
   ---------------------------------- ----- 86.8/101.7 MB 68.4 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 68.9 MB/s eta 0:00:01
   ---------------------------------------- 101.7/101.7 MB 60.7 MB/s  0:00:01


## 7. Registro del modelo final en experiments_log.csv

Se agrega el modelo final al registro de experimentos del proyecto, indicando que corresponde al modelo seleccionado al cierre del Sprint 4.

In [38]:
result_final = {
    "model": "final_model",
    "params": get_model_params(final_model),
    "date": datetime.today().strftime("%Y-%m-%d"),

    "cv_accuracy": None,
    "cv_precision": None,
    "cv_recall": None,
    "cv_f1": None,
    "cv_roc_auc": None,

    "test_accuracy": round(final_metrics["test_accuracy"], 6),
    "test_recall": round(final_metrics["test_recall"], 6),
    "test_f1": round(final_metrics["test_f1"], 6),
    "test_roc_auc": round(final_metrics["test_roc_auc"], 6) if final_metrics["test_roc_auc"] is not None else None,

    "recall_gap": None,
    "f1_gap": None,
    "train_time_s": None,

    "selected": True,
    "notes": "Modelo final Sprint 4 validado y registrado por Experiment Tracker",

    "tipo_modelo": "final",
    "dataset": "global",
    "path": FINAL_MODEL_PATH
}

experiments_log_updated = save_experiment(result_final, LOG_PATH)

experiments_log_updated.tail()

⚠️ Este modelo ya estaba registrado en experiments_log.csv


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
11,rf,REEMPLAZAR_CON_BEST_PARAMS_RF,2026-05-08,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Random Forest tuneado en Sprint 4,tuned,global,../models/tuned_rf.pkl
12,xgb,REEMPLAZAR_CON_BEST_PARAMS_XGB,2026-05-08,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,XGBoost tuneado en Sprint 4,tuned,global,../models/tuned_xgb.pkl
13,voting_rf_xgb,soft voting: tuned_rf + tuned_xgb,2026-05-08,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Ensamble Soft Voting con RF tuneado y XGB tuneado,ensemble,global,../models/ensemble_voting_rf_xgb.pkl
14,stacking_rf_xgb,base models: tuned_rf + tuned_xgb; meta learne...,2026-05-08,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Stacking con RF tuneado y XGB tuneado; Logisti...,ensemble,global,../models/ensemble_stacking_rf_xgb.pkl
15,final_model,REEMPLAZAR_CON_MODELO_FINAL_SELECCIONADO,2026-05-08,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,True,Modelo final Sprint 4 validado en test set,final,global,../models/final_model.pkl


## 8. Resumen del registro actualizado

Se presenta una vista resumida del registro de experimentos, mostrando los modelos baseline y el modelo final registrado durante el Sprint 4.

In [39]:
experiments_log_updated = pd.read_csv(LOG_PATH)

cols_show = [
    "model", "tipo_modelo", "dataset",
    "cv_recall", "cv_f1", "cv_roc_auc",
    "test_accuracy", "test_recall", "test_f1", "test_roc_auc",
    "selected", "path", "notes"
]

available_cols = [col for col in cols_show if col in experiments_log_updated.columns]

experiments_log_updated[available_cols]

,model,tipo_modelo,dataset,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_recall,test_f1,test_roc_auc,selected,path,notes
0,dt,NaN,NaN,0.761476,0.754286,0.807445,0.821484,0.773278,0.768712,0.820202,True,NaN,Rank 1 CV Recall. Sin overfitting. Seleccionad...
1,rf,NaN,NaN,0.752171,0.802576,0.926593,0.866994,0.769205,0.815815,0.934606,True,NaN,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...
2,xgb,NaN,NaN,0.713676,0.775089,0.915453,0.845560,0.716031,0.776850,0.916261,True,NaN,Rank 3 CV Recall. Gap mínimo de overfitting. E...
3,lr,NaN,NaN,0.618413,0.709945,0.860963,0.784866,0.615115,0.706379,0.863555,False,NaN,CV Recall bajo (0.618). Modelo lineal insufici...
4,gb,NaN,NaN,0.616122,0.715516,0.885453,0.814479,0.611495,0.711231,0.883982,False,NaN,CV Recall bajo (0.616). Más lento que RF/XGB s...
5,NN,NaN,NaN,0.614821,0.703766,0.873830,NaN,0.647585,0.726027,0.885184,False,NaN,CV Recall más bajo (0.615). Mayor tiempo de en...
6,tuned_rf,NaN,NaN,0.713400,0.785000,0.924300,NaN,0.724100,0.793300,0.930700,False,NaN,RF tuneado con RandomizedSearchCV. Mejora reca...
7,tuned_xgb,NaN,NaN,0.766300,0.804100,0.927300,NaN,0.775300,0.812500,0.935700,True,NaN,XGB tuneado con RandomizedSearchCV. MODELO FIN...
8,ensemble_hv,NaN,NaN,0.692300,0.780600,NaN,NaN,0.708300,0.792900,NaN,False,NaN,"Hard Voting. No soporta predict_proba, AUC no ..."
9,ensemble_sv,NaN,NaN,0.749500,0.802600,0.929900,NaN,0.760300,0.812800,0.937000,False,NaN,Soft Voting con pesos iguales. AUC más alto de...


## 9. Nota sobre modelos versionados con DVC

Algunos modelos tuneados y ensamblados se encuentran gestionados mediante DVC y aparecen como archivos `.dvc`. Debido a restricciones de autenticación del remote externo, en este entorno local se validó directamente el archivo `final_model.pkl`, correspondiente al modelo final seleccionado del Sprint 4.

El registro mantiene la trazabilidad del Sprint 3 y agrega la validación del modelo final disponible localmente.

## Conclusión

Se actualizó el archivo `experiments_log.csv` incorporando el modelo final del Sprint 4.  
El modelo final fue cargado correctamente con `joblib` y generó predicciones sobre el conjunto de prueba.  

Con ello, el rol de Experiment Tracker consolida la trazabilidad del proyecto desde los modelos baseline del Sprint 3 hasta el modelo final optimizado y validado del Sprint 4.